# Experimentos B, C, D — ResNet-50 y Xception (en Colab)

Toma los modelos entrenados en el Experimento A y los evalúa, **sin reentrenar**

| Experimento | Se evalúa en | Mide |
|---|---|---|
| **B** | StyleGAN3 | generalización dentro de los GAN |
| **C** | SDXL | generalización entre familias (GAN → difusión) |
| **D** | Flux | generalización de difusión mas alejada del entrenamiento |

### Antes de correr, sube a Drive:
```
MyDrive/Tesis/
    datos/config.py, nucleo.py        <-- codigos utilizados en el experimento anterior
    datos/StyleGAN3.zip,datos/SDXL.zip
    datos/Flux.zip                    <-- Datasets a utilizar para estos experimentos: B/C/D
    Resultados/modelos/*.pth          <-- que se encuentran en la carpeta de resultados obtenidos del Experimento A
```
Cada zip, al descomprimirse, debe dar una carpeta `<Generador>/test/fake` y `<Generador>/test/real`.
(Solo se usa el split `test`, pero si el zip trae train/val también, no afecta en la ejecución.)

In [ ]:
!nvidia-smi

In [ ]:
!pip install timm --quiet
print("Listo")

In [ ]:
from google.colab import drive
import os, time, zipfile, glob, shutil

drive.mount('/content/drive')

RUTA_DATOS_DRIVE = '/content/drive/MyDrive/Tesis/datos'   # <-- AJUSTA si cambia
RUTA_DATOS_LOCAL = '/content/Particiones'

ZIP_DE_GENERADOR = {
    "StyleGAN2_CelebA": "StyleGAN2_CelebA.zip",   # <-- AJUSTA el nombre si difiere
    "StyleGAN3_CelebA": "StyleGAN3_CelebA.zip",
    "SDXL_CelebA":      "SDXL_Celeb_A.zip",
    "Flux_CelebA":      "Flux_Celeb_A.zip",
}
GENERADORES_EVAL = list(ZIP_DE_GENERADOR)

os.makedirs(RUTA_DATOS_LOCAL, exist_ok=True)

for gen, zipname in ZIP_DE_GENERADOR.items():
    destino = os.path.join(RUTA_DATOS_LOCAL, gen)
    if os.path.isdir(os.path.join(destino, "test")):
        print(f"[{gen}] ya descomprimido")
        continue

    ruta_zip = os.path.join(RUTA_DATOS_DRIVE, zipname)
    if not os.path.exists(ruta_zip):
        raise FileNotFoundError(
            f"No encuentro {ruta_zip}\n"
            f"Revisa el nombre exacto en Drive (mayúsculas/guiones incluidos)."
        )

    print(f"[{gen}] descomprimiendo {zipname}...")
    t0 = time.time()
    tmp = os.path.join(RUTA_DATOS_LOCAL, f"_tmp_{gen}")
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(ruta_zip) as z:
        z.extractall(tmp)
    print(f"[{gen}] extraído en {time.time()-t0:.0f}s")

    # Buscar dónde quedó el split 'test' dentro de lo extraído
    cand = glob.glob(os.path.join(tmp, "**", "test"), recursive=True)
    if not cand:
        cand_fake = glob.glob(os.path.join(tmp, "**", "fake"), recursive=True)
        raise FileNotFoundError(
            f"[{gen}] tras descomprimir no encuentro una carpeta 'test'.\n"
            f"Contenido de nivel superior: {os.listdir(tmp)}\n"
            f"(carpetas 'fake' encontradas: {cand_fake[:3]})"
        )
    carpeta_con_test = os.path.dirname(cand[0])
    shutil.rmtree(destino, ignore_errors=True)
    shutil.move(carpeta_con_test, destino)
    shutil.rmtree(tmp, ignore_errors=True)

print("\n" + "=" * 66)
print("CONTEO POR GENERADOR (test)")
print("=" * 66)
for gen in GENERADORES_EVAL:
    for clase in ["fake", "real"]:
        carpeta = os.path.join(RUTA_DATOS_LOCAL, gen, "test", clase)
        n = len(os.listdir(carpeta)) if os.path.isdir(carpeta) else 0
        estado = "" if n > 0 else "  <-- VACÍA O AUSENTE"
        print(f"  {gen:16}/test/{clase}: {n:>5} imágenes{estado}")

In [ ]:
import shutil, sys
from pathlib import Path

RUTA_CODIGO_DRIVE = Path('/content/drive/MyDrive/Tesis/codigo')  # <-- AJUSTA

for nombre in ['config.py', 'nucleo.py']:
    origen = RUTA_CODIGO_DRIVE / nombre
    if not origen.exists():
        raise FileNotFoundError(f"No encuentro {origen}. Súbelo a {RUTA_CODIGO_DRIVE}/")
    shutil.copy(origen, Path('/content') / nombre)

sys.path.insert(0, '/content')
import config
import nucleo

config.RUTA_PARTICIONES = Path(RUTA_DATOS_LOCAL)
config.RUTA_RESULTADOS  = Path('/content/drive/MyDrive/Tesis/Resultados')
config.RUTA_MODELOS     = config.RUTA_RESULTADOS / "modelos"
config.RUTA_METRICAS    = config.RUTA_RESULTADOS / "metricas"
config.NUM_WORKERS = 4
config.preparar_carpetas()

print("Motor importado. Modelos se leerán de:", config.RUTA_MODELOS)

In [ ]:
import json, statistics
from datetime import datetime

import timm
import torch

ARQUITECTURAS = ["efficientnet_b0", "resnet50", "legacy_xception"]
GENERADOR_ENTRENAMIENTO = "StyleGAN2_CelebA"
EXPERIMENTOS = {"B": "StyleGAN3_CelebA", "C": "SDXL_CelebA", "D": "Flux_CelebA"}


def cargar_acc_A(arquitectura):
    """Lee el .json del Experimento A (ya en Drive) para la degradación."""
    ruta = config.RUTA_METRICAS / f"experimentoA_{arquitectura}_{GENERADOR_ENTRENAMIENTO}.json"
    if not ruta.exists():
        print(f"  AVISO: no encuentro {ruta.name}; sin degradación para {arquitectura}.")
        return {}
    with open(ruta, encoding="utf-8") as f:
        datos = json.load(f)
    return {r["semilla"]: r["metricas_test"]["accuracy"]
            for r in datos["resultados_por_semilla"]}


for arquitectura in ARQUITECTURAS:
    print("=" * 64)
    print(f"EXPERIMENTOS B, C, D — {arquitectura}")
    print("=" * 64)

    acc_A = cargar_acc_A(arquitectura)
    resultados = []
    faltantes = []

    for semilla in config.SEMILLAS_ENTRENAMIENTO:
        nombre_pth = f"{arquitectura}_{GENERADOR_ENTRENAMIENTO}_semilla{semilla}.pth"
        ruta_pth = config.RUTA_MODELOS / nombre_pth
        if not ruta_pth.exists():
            faltantes.append(nombre_pth)
            continue

        modelo = timm.create_model(arquitectura, pretrained=False,
                                   num_classes=config.NUM_CLASES).to(config.DISPOSITIVO)
        modelo.load_state_dict(torch.load(ruta_pth, map_location=config.DISPOSITIVO))
        modelo.eval()

        for exp, generador in EXPERIMENTOS.items():
            loader = nucleo.crear_dataloader(generador, "test", semilla, barajar=False)
            m = nucleo.evaluar(modelo, loader)

            registro = {
                "experimento": exp,
                "arquitectura": arquitectura,
                "semilla": semilla,
                "generador_evaluacion": generador,
                "metricas": m,
            }
            if semilla in acc_A:
                registro["acc_experimento_A"] = acc_A[semilla]
                registro["degradacion_acc"] = acc_A[semilla] - m["accuracy"]
            resultados.append(registro)

            deg = registro.get("degradacion_acc")
            deg_txt = f" | degradación: {deg:+.4f}" if deg is not None else ""
            print(f"  [{exp}] semilla {semilla} en {generador}: "
                  f"acc {m['accuracy']:.4f} | auc {m['auc']:.4f}{deg_txt}")

    if faltantes:
        print("\n  AVISO: faltan estos modelos en Drive (¿corriste el Experimento A?):")
        for f_ in faltantes:
            print(f"    - {f_}")

    salida = {
        "experimentos": "B_C_D_cross_generador",
        "arquitecturas": [arquitectura],
        "generador_entrenamiento": GENERADOR_ENTRENAMIENTO,
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "evaluaciones": resultados,
    }
    ruta_json = config.RUTA_METRICAS / f"experimentosBCD_{arquitectura}.json"
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(salida, f, indent=2, ensure_ascii=False)

    print(f"\n  {'':<10} | {'B (SG3)':>9} | {'C (SDXL)':>9} | {'D (Flux)':>9}")
    fila = f"  {'acc medio':<10}"
    for exp in EXPERIMENTOS:
        accs = [r["metricas"]["accuracy"] for r in resultados if r["experimento"] == exp]
        fila += f" | {statistics.mean(accs):>9.4f}" if accs else f" | {'--':>9}"
    print(fila)
    print(f"  -> {ruta_json}\n")

print("=" * 64)
print("TERMINADO — B, C, D para las tres arquitecturas")
print("=" * 64)

In [ ]:
import os
base = "/content/drive/MyDrive/Tesis"
for sub in ["Resultados", "Resultados2"]:
    d = os.path.join(base, sub, "modelos")
    print(f"\n=== {sub}/modelos ===")
    if os.path.isdir(d):
        for f in sorted(os.listdir(d)):
            print("  ", f)
    else:
        print("   (no existe)")

In [ ]:
import os, hashlib
base = "/content/Particiones"
def h(gen):
    d = os.path.join(base, gen, "test", "real")
    f = sorted(os.listdir(d))
    return len(f), hashlib.md5(open(os.path.join(d, f[0]),"rb").read()).hexdigest()
for g in ["StyleGAN2_CelebA", "StyleGAN3_CelebA", "SDXL_CelebA", "Flux_CelebA"]:
    print(g, h(g))

---
## Cuando termine

Baja de Drive `Tesis/Resultados/metricas/experimentosBCD_resnet50.json` y `experimentosBCD_legacy_xception.json`
a tu `Resultados\metricas\` local.

Con eso ya tienes en local los `.json` de A (los tres) y de B/C/D (los tres). Solo falta:
```
python analizar_resultados.py
```
para generar todas las tablas y figuras con las tres arquitecturas juntas.

